# Detection — The 4-Step Fallback Chain

This notebook walks through **why** `app.core.extraction.detect_grid` is a 4-step deterministic fallback chain (rather than a single-pass contour detector), and **how** each step was chosen.

**Prerequisites.** Full dev requirements (`pip install -r ../requirements.txt`) and the 38-image ground-truth set at `../Examples/Ground Example/` (committed to the repo, ~2 MB).

**Related reading**

- [`02_scoring.ipynb`](02_scoring.ipynb) — derivation of the 5-component scoring function used in Step 1
- `../app/core/extraction.py:597` — the production implementation of `detect_grid`

In [ ]:
import sys
import json
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve().parent  # notebooks/ → repo root
sys.path.insert(0, str(PROJECT_ROOT))

from app.core.extraction import (
    detect_grid,
    find_grid_contour,
    _preprocess,
    _find_best_quad_structured,
    _find_quad_standard,
    order_points,
)

GT_PATH = PROJECT_ROOT / "evaluation" / "ground_truth.json"
IMAGES_DIR = PROJECT_ROOT / "Examples" / "Ground Example"

with open(GT_PATH) as f:
    gt_entries = json.load(f)["images"]

print(f"ground-truth entries: {len(gt_entries)}")
print(f"images directory present: {IMAGES_DIR.exists()}")


## 1. The problem

A "detection" in this project means: take a BGR photo of a newspaper page and return the four outer corners of the Sudoku grid (ordered `[TL, TR, BR, BL]`), sub-pixel refined, with a confidence score.

The *easy* images are trivial — a clean photo with a centered grid solves with almost any contour-based recipe. The interesting failure modes come from real newsprint:

| Image | Failure mode |
|---|---|
| `_33_` | Sudoku grid sits *inside* a crossword block; `RETR_EXTERNAL` picks the crossword |
| `_17_` | Faint faded print; default CLAHE (`clipLimit=2.0`) can't recover the lines |
| `_23_`, `_26_` | Outer border broken by paper creases; closed quad doesn't exist after thresholding |
| `_39_` | Fragmented outer contour; needs net-thinning morphology to close gaps |
| `_1_`, `_21_`, `_35_`, `_38_` | Unrecoverable in this family (too low-contrast / too occluded) |

No single preprocessing recipe handles all of these. A **sequence** of detectors, each targeting a specific failure mode, does.


In [ ]:
def load_rgb(name):
    img = cv2.imread(str(IMAGES_DIR / name))
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if img is not None else None

easy = load_rgb("_22_7288515.jpeg")       # clean case
hard = load_rgb("_33_3803709.jpeg")        # crossword-nested

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(easy); axes[0].set_title("Easy: clean newspaper photo"); axes[0].axis("off")
axes[1].imshow(hard); axes[1].set_title("Hard: Sudoku nested inside a crossword"); axes[1].axis("off")
plt.tight_layout(); plt.show()


## 2. Why single-pass detection fails

The first detector written in this project was a single-pass pipeline: CLAHE → Gaussian blur → adaptive threshold → `findContours(RETR_EXTERNAL)` → pick the largest 4-vertex convex quadrilateral. It worked on clean synthetic images and failed on real newspaper photos — the commit history has a blunt note about this (`713d6ef`, 2026-04-03): *"contour 100% on clean → 0% on real."*

`RETR_EXTERNAL` only returns the topmost contour in each connected region, which means on `_33_` it returns the crossword block (the larger outer contour) and never sees the Sudoku grid nested inside it. Let's reproduce the failure:


In [ ]:
# Single-pass baseline: Step 1 preprocessing + find_grid_contour (RETR_EXTERNAL)
gray = cv2.cvtColor(cv2.cvtColor(hard, cv2.COLOR_RGB2BGR), cv2.COLOR_BGR2GRAY)
thresh = _preprocess(gray)
contour = find_grid_contour(thresh)

overlay = hard.copy()
if contour is not None:
    pts = contour.reshape(-1, 2).astype(int)
    cv2.polylines(overlay, [pts], isClosed=True, color=(255, 0, 0), thickness=4)

plt.figure(figsize=(7, 7))
plt.imshow(overlay)
plt.title("Single-pass (RETR_EXTERNAL): picks the crossword, not the Sudoku")
plt.axis("off")
plt.show()


## 3. Step 1 — RETR_TREE + structure-aware scoring

The fix for `_33_` has two parts:

1. **Switch to `cv2.RETR_TREE`** so the contour walker sees *every* contour in the hierarchy, including nested ones.
2. **Score each candidate** on how grid-like its *interior* looks, not just its outer shape. A quad that's square and centered but contains crossword text should lose to a smaller quad that contains a 9×9 grid pattern.

The 5-component score is:

```
score  =  0.20 · area_norm
       +  0.20 · squareness
       +  0.10 · centeredness
       +  0.30 · grid_structure      ← derived in 02_scoring.ipynb
       +  0.20 · cell_count          ← derived in 02_scoring.ipynb
```

The two structure-aware components (`grid_structure`, `cell_count`) are what make Step 1 reject look-alike candidates. Their full derivation is in [`02_scoring.ipynb`](02_scoring.ipynb). Here's what Step 1 picks on `_33_`:


In [ ]:
# Run the full production detect_grid — Step 1 handles this image
corners, confidence = detect_grid(cv2.cvtColor(hard, cv2.COLOR_RGB2BGR))

overlay = hard.copy()
if corners is not None:
    pts = corners.astype(int)
    cv2.polylines(overlay, [pts], isClosed=True, color=(0, 255, 0), thickness=4)
    for label, (x, y) in zip(["TL", "TR", "BR", "BL"], pts):
        cv2.circle(overlay, (int(x), int(y)), 10, (255, 255, 0), -1)
        cv2.putText(overlay, label, (int(x)+14, int(y)-10), cv2.FONT_HERSHEY_SIMPLEX,
                    0.8, (255, 255, 0), 2)

plt.figure(figsize=(7, 7))
plt.imshow(overlay)
plt.title(f"detect_grid (Step 1 = RETR_TREE + structure scoring)  confidence={confidence:.2f}")
plt.axis("off")
plt.show()


## 4. Steps 2–4 — preprocessing fallbacks for specific failure modes

Step 1 handles **29 of the 34 successful detections** on the 38-image ground-truth set. The remaining 5 images need different preprocessing because their grid lines are fragmented, faint, or broken. Steps 2–4 are simpler fallbacks — they use `cv2.RETR_EXTERNAL` and the older 3-component score `0.4·area + 0.3·squareness + 0.3·centeredness`, but each step applies a different preprocessing recipe targeted at a specific real-world failure mode.

| Step | Preprocessing delta vs Step 1 | Rescues images | Rationale |
|---|---|---|---|
| 2 | `dilate(3×3)` then `erode(5×5)` (net thinning) | `_39_` | Closes small gaps in fragmented outer contours without fattening grid lines |
| 3 | **CLAHE `clipLimit=6.0`**, threshold `C=7` | `_17_`, `_24_` | Aggressive local-contrast enhancement for faint or faded print |
| 4 | `dilate(3×3)` then `erode(3×3)` (symmetric closing) | `_23_`, `_26_` | Morphological closing for broken lines from ink bleed or paper creases |

The fallback order is important. Step 3's aggressive CLAHE fattens dark regions, which fixes faint print but introduces false contours on normal images — so it runs only after Steps 1 and 2 have failed. Step 4's symmetric closing kernel is different enough from Step 2's net-thinning recipe that the two cover distinct failure patterns without overlapping.

**Worst-case latency** is all four steps attempted: ~15 ms. **Typical case** (Step 1 succeeds): ~5 ms. Both measured by `evaluation/evaluate_detection.py` on the GT set.


In [ ]:
# Demonstrate Step 3 rescuing _17_ (faint print)
faint = cv2.imread(str(IMAGES_DIR / "_17_3814183.jpeg"))
if faint is None:
    print("skip: _17_ not available in Examples/Ground Example/")
else:
    faint_rgb = cv2.cvtColor(faint, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(faint, cv2.COLOR_BGR2GRAY)
    thresh1 = _preprocess(gray)                               # Step 1 preprocessing
    thresh3 = _preprocess(gray, clahe_clip=6.0, thresh_c=7)   # Step 3 preprocessing

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(faint_rgb);         axes[0].set_title("_17_  (faint print input)"); axes[0].axis("off")
    axes[1].imshow(thresh1, cmap="gray"); axes[1].set_title("Step 1 preprocessing (CLAHE clip=2.0)"); axes[1].axis("off")
    axes[2].imshow(thresh3, cmap="gray"); axes[2].set_title("Step 3 preprocessing (CLAHE clip=6.0, C=7)"); axes[2].axis("off")
    plt.tight_layout(); plt.show()

## 5. Running the full chain against the ground-truth set

The chain is deterministic and has no tunable parameters, so one pass over all 38 images gives a definitive detection rate.


In [ ]:
detected = []
missed = []
for entry in gt_entries:
    name = Path(entry["path"]).name
    img = cv2.imread(str(IMAGES_DIR / name))
    if img is None:
        continue
    corners, _ = detect_grid(img)
    (detected if corners is not None else missed).append(name)

print(f"Detection rate: {len(detected)}/{len(gt_entries)} ({100 * len(detected)/len(gt_entries):.1f}%)")
print(f"Missed ({len(missed)}): {missed}")


## 6. The 4 irrecoverable images

Four images are missed by all four steps of the chain: `_1_`, `_21_`, `_35_`, `_38_`. Their grid edges are too low-contrast or too occluded for any single-pass contour detector in this family to recover. Fixing them would require either:

1. **A different detection paradigm** — a learned grid detector (e.g., a small CNN or line-segment detector trained on the 34 successful cases).
2. **Manual corner annotation** — the `/debug` endpoint in the web frontend already supports draggable corner handles for exactly this case.

The user-facing debug route uses `find_grid_contour` directly with tunable preprocessing parameters, so failing images can be debugged interactively by dragging corners onto the actual grid and re-running the pipeline with the manual quad.

## 7. References

- `app/core/extraction.py:597` — `detect_grid` implementation
- [`02_scoring.ipynb`](02_scoring.ipynb) — the 5-component scoring function, derived cell by cell
- `evaluation/evaluate_detection.py` — production detector benchmark (34/38, median corner error 1.6 px, median IoU 0.99)
- **[1]** Kamal, Chawla & Goel, *Detection of Sudoku Puzzle using Image Processing and Solving by Backtracking, Simulated Annealing and Genetic Algorithms* (ICIIP 2015) — the image-processing + backtracking reference cited in the main README